# 전 자세 목각인형 검출기 학습

서기·앉기·눕기를 모두 `mannequin` 클래스 하나로 검출하는 YOLO11n 모델을 학습합니다. 검출 모델은 자세를 판정하지 않고 bbox만 만들며, 자세 판정은 이후 `detect_then_pose_webcam_test.py`의 crop Pose 단계가 담당합니다.

## Drive에 준비할 파일

- `rescue_dataset.zip`: 기존 YOLO Detection 데이터셋. class 0=`fallen_person`, class 1=`helper_rc_car`
- `standing_dataset.zip`: 서기·앉기 목각인형을 **class 0 bbox로 직접 라벨링한** YOLO Detection 데이터셋
- `rescue_yolo11n.pt`(선택): 기존 누운 목각인형 모델에서 이어 학습할 때 사용

> 기존 hard-negative의 빈 txt 파일에는 bbox 좌표가 없으므로 자동으로 standing 라벨로 바꿀 수 없습니다. 서 있는 목각인형 사진은 반드시 bbox를 새로 그려야 합니다. 진짜 빈 바닥만 빈 라벨로 유지합니다.

ZIP 내부는 `images/train`, `images/val`, `images/test`, `labels/train`, `labels/val`, `labels/test` 구조여야 합니다.

In [ ]:
!nvidia-smi
!pip -q install "ultralytics>=8.3,<9" pyyaml

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/mannequin_training')
RESCUE_ZIP = DRIVE_DIR / 'rescue_dataset.zip'
STANDING_ZIP = DRIVE_DIR / 'standing_dataset.zip'
INITIAL_WEIGHTS = DRIVE_DIR / 'rescue_yolo11n.pt'
OUTPUT_DIR = DRIVE_DIR / 'runs'

for path in (RESCUE_ZIP, STANDING_ZIP):
    if not path.is_file():
        raise FileNotFoundError(f'Drive에 필요한 파일이 없습니다: {path}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('기존 데이터:', RESCUE_ZIP)
print('정상 자세 데이터:', STANDING_ZIP)
print('초기 가중치:', INITIAL_WEIGHTS if INITIAL_WEIGHTS.is_file() else 'yolo11n.pt (COCO)')

## 데이터 병합

기존 class 0 `fallen_person`은 새 데이터셋에서 class 0 `mannequin`으로 의미만 변경합니다. 서기·앉기 데이터도 class 0을 사용합니다. RC카는 class 1을 유지합니다. 서로 다른 촬영 묶음이 train/val/test에 중복되지 않도록 ZIP을 만들기 전에 분리해야 합니다.

In [ ]:
import shutil
import zipfile
from collections import Counter

WORK = Path('/content/mannequin_training')
EXTRACT = WORK / 'extracted'
MERGED = WORK / 'dataset'
if WORK.exists():
    shutil.rmtree(WORK)
EXTRACT.mkdir(parents=True)

def extract_zip(source: Path, name: str) -> Path:
    target = EXTRACT / name
    target.mkdir()
    with zipfile.ZipFile(source) as archive:
        archive.extractall(target)
    candidates = [p.parent for p in target.rglob('images') if (p / 'train').is_dir()]
    if len(candidates) != 1:
        raise RuntimeError(f'{source.name}: images/train을 가진 데이터 루트가 1개여야 합니다: {candidates}')
    root = candidates[0]
    for split in ('train', 'val', 'test'):
        for kind in ('images', 'labels'):
            if not (root / kind / split).is_dir():
                raise RuntimeError(f'{source.name}: {kind}/{split}이 없습니다.')
    return root

rescue_root = extract_zip(RESCUE_ZIP, 'rescue')
standing_root = extract_zip(STANDING_ZIP, 'standing')
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def validate_and_copy(root: Path, prefix: str, require_positive: bool):
    stats = Counter()
    for split in ('train', 'val', 'test'):
        image_out = MERGED / 'images' / split
        label_out = MERGED / 'labels' / split
        image_out.mkdir(parents=True, exist_ok=True)
        label_out.mkdir(parents=True, exist_ok=True)
        images = [p for p in (root / 'images' / split).iterdir() if p.suffix.lower() in IMAGE_EXTS]
        for image in images:
            label = root / 'labels' / split / f'{image.stem}.txt'
            if not label.is_file():
                raise RuntimeError(f'라벨 누락: {label}')
            lines = [line.strip() for line in label.read_text().splitlines() if line.strip()]
            for line in lines:
                fields = line.split()
                if len(fields) != 5:
                    raise RuntimeError(f'YOLO Detection 5필드가 아닙니다: {label}: {line}')
                class_id = int(fields[0])
                if class_id not in (0, 1):
                    raise RuntimeError(f'허용하지 않는 class {class_id}: {label}')
                values = [float(value) for value in fields[1:]]
                if any(value < 0.0 or value > 1.0 for value in values):
                    raise RuntimeError(f'정규화 범위를 벗어난 bbox: {label}: {line}')
                stats[f'class_{class_id}'] += 1
            stats[f'{split}_images'] += 1
            stats['positive_images' if lines else 'empty_images'] += 1
            new_stem = f'{prefix}_{image.stem}'
            shutil.copy2(image, image_out / f'{new_stem}{image.suffix.lower()}')
            (label_out / f'{new_stem}.txt').write_text(('\n'.join(lines) + '\n') if lines else '')
    if require_positive and stats['class_0'] == 0:
        raise RuntimeError(f'{prefix} 데이터에 class 0 bbox가 하나도 없습니다. 빈 hard-negative만 올린 것은 아닌지 확인하세요.')
    return stats

rescue_stats = validate_and_copy(rescue_root, 'rescue', require_positive=True)
standing_stats = validate_and_copy(standing_root, 'standing', require_positive=True)
print('기존 데이터:', dict(rescue_stats))
print('정상 자세 데이터:', dict(standing_stats))
print('통합 위치:', MERGED)

In [ ]:
import yaml

data_yaml = WORK / 'mannequin_data.yaml'
data_yaml.write_text(yaml.safe_dump({
    'path': str(MERGED),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {0: 'mannequin', 1: 'helper_rc_car'},
}, sort_keys=False, allow_unicode=True))
print(data_yaml.read_text())

In [ ]:
import random
import cv2
import matplotlib.pyplot as plt

def draw_labels(image_path: Path):
    split = image_path.parent.name
    label_path = MERGED / 'labels' / split / f'{image_path.stem}.txt'
    image = cv2.imread(str(image_path))
    height, width = image.shape[:2]
    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue
        class_id, cx, cy, bw, bh = map(float, line.split())
        x1, y1 = int((cx - bw / 2) * width), int((cy - bh / 2) * height)
        x2, y2 = int((cx + bw / 2) * width), int((cy + bh / 2) * height)
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(image, 'mannequin' if int(class_id) == 0 else 'helper_rc_car',
                    (x1, max(y1 - 5, 18)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2)
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

standing_images = list((MERGED / 'images' / 'train').glob('standing_*'))
samples = random.sample(standing_images, min(8, len(standing_images)))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for axis in axes.flat:
    axis.axis('off')
for axis, sample in zip(axes.flat, samples):
    axis.imshow(draw_labels(sample))
    axis.set_title(sample.name[:35])
plt.tight_layout()
plt.show()
if not samples:
    raise RuntimeError('standing 학습 이미지가 없습니다.')

## YOLO11n 학습

기존 `rescue_yolo11n.pt`가 Drive에 있으면 이를 초기 가중치로 사용해 누운 목각인형과 RC카 특징을 보존합니다. 없으면 COCO `yolo11n.pt`에서 시작합니다. T4 GPU 메모리가 부족하면 `BATCH=8` 또는 `4`로 낮춥니다.

In [ ]:
import torch
from ultralytics import YOLO

EPOCHS = 100
BATCH = 16
IMGSZ = 640
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
start_weights = str(INITIAL_WEIGHTS) if INITIAL_WEIGHTS.is_file() else 'yolo11n.pt'
model = YOLO(start_weights)
model.train(
    data=str(data_yaml), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    device=DEVICE, workers=2, patience=20, seed=42, deterministic=True,
    project=str(OUTPUT_DIR), name='mannequin_yolo11n', plots=True,
    scale=0.7, degrees=15.0, translate=0.15, fliplr=0.5,
    close_mosaic=10, exist_ok=True,
)

In [ ]:
run_dir = OUTPUT_DIR / 'mannequin_yolo11n'
best = run_dir / 'weights' / 'best.pt'
if not best.is_file():
    raise FileNotFoundError(best)
best_model = YOLO(str(best))
print('classes:', best_model.names)
val_result = best_model.val(data=str(data_yaml), split='val', imgsz=IMGSZ, device=DEVICE, plots=True)
test_result = best_model.val(data=str(data_yaml), split='test', imgsz=IMGSZ, device=DEVICE, plots=True, name='mannequin_yolo11n_test')
print('best:', best)
print('test mAP50:', float(test_result.box.map50))
print('test mAP50-95:', float(test_result.box.map))

In [ ]:
from google.colab import files

final_weights = DRIVE_DIR / 'mannequin_yolo11n_best.pt'
shutil.copy2(best, final_weights)
archive_base = Path('/content/mannequin_yolo11n_results')
archive = shutil.make_archive(str(archive_base), 'zip', run_dir)
print('Drive 가중치:', final_weights)
print('결과 ZIP:', archive)
files.download(str(best))
files.download(archive)

## 로컬 웹캠 테스트

다운로드한 `best.pt`를 프로젝트 PC로 복사한 뒤 다음처럼 실행합니다. `--target mannequin`은 기본 모델 선택과 화면 이름을 지정하며, 실제 detector는 `--detector-weights`에 준 새 모델을 사용합니다.

```bash
/usr/bin/python3 vision_training/testing/detect_then_pose_webcam_test.py \
  --source 0 \
  --target mannequin \
  --detector-weights /absolute/path/mannequin_yolo11n_best.pt \
  --det-conf 0.25 --pose-conf 0.10 --keypoint-conf 0.15
```

평가할 때는 객체 bbox가 나왔는지와 Pose 관절이 나왔는지를 따로 기록합니다. bbox는 나오지만 관절이 나오지 않는다면 1단계 mannequin 검출은 해결됐고 2단계 사람용 Pose의 도메인 한계가 남은 것입니다.